In [0]:
exec(open("/Workspace/ecommerce_retail/config.py").read())

df_inventory = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/inventory_health/")

df_inventory.createOrReplaceTempView("inventory_health")
print(f"✅ {df_inventory.count()} rows loaded")
df_inventory.printSchema()

✅ Credentials loaded (use as .option() in read/write calls)
✅ 10000 rows loaded
root
 |-- inventory_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- warehouse_city: string (nullable = true)
 |-- stock_on_hand: long (nullable = true)
 |-- stock_reserved: long (nullable = true)
 |-- available_stock: long (nullable = true)
 |-- reorder_point: long (nullable = true)
 |-- inventory_value: double (nullable = true)
 |-- inventory_status: string (nullable = true)
 |-- low_stock_flag: boolean (nullable = true)
 |-- etl_timestamp: timestamp (nullable = true)
 |-- warehouse_country: string (nullable = true)



In [0]:
!pip install plotly pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [0]:
df = spark.sql("""
    SELECT
        COUNT(DISTINCT product_id)                        AS total_products,
        SUM(stock_on_hand)                                AS total_stock_on_hand,
        SUM(stock_reserved)                               AS total_stock_reserved,
        SUM(available_stock)                              AS total_available_stock,
        ROUND(SUM(inventory_value), 2)                    AS total_inventory_value,
        COUNT(CASE WHEN low_stock_flag = true THEN 1 END) AS low_stock_products,
        COUNT(CASE WHEN inventory_status = 'out_of_stock'
                    OR stock_on_hand = 0 THEN 1 END)      AS out_of_stock_products
    FROM inventory_health
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="📦 Inventory Health — KPI Summary")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        inventory_status,
        COUNT(product_id) AS product_count
    FROM inventory_health
    GROUP BY inventory_status
    ORDER BY product_count DESC
""").toPandas()

fig = px.pie(
    df,
    names="inventory_status",
    values="product_count",
    title="🟢 Stock Status Breakdown",
    hole=0.4,
    color="inventory_status",
    color_discrete_map={
        "in_stock":     "#2ECC71",
        "low_stock":    "#F39C12",
        "out_of_stock": "#E74C3C",
        "overstocked":  "#3498DB"
    }
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        warehouse_country,
        ROUND(SUM(inventory_value), 2) AS total_inventory_value,
        COUNT(DISTINCT product_id)     AS unique_products,
        SUM(stock_on_hand)             AS total_stock
    FROM inventory_health
    GROUP BY warehouse_country
    ORDER BY total_inventory_value DESC
""").toPandas()

fig = px.bar(
    df,
    x="warehouse_country",
    y="total_inventory_value",
    title="🌍 Total Inventory Value by Country",
    color="warehouse_country",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="total_inventory_value"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Country",
    yaxis_title="Inventory Value ($)"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        warehouse_country,
        COUNT(CASE WHEN low_stock_flag = true  THEN 1 END) AS low_stock,
        COUNT(CASE WHEN low_stock_flag = false THEN 1 END) AS healthy_stock
    FROM inventory_health
    GROUP BY warehouse_country
    ORDER BY warehouse_country
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df["warehouse_country"],
    y=df["healthy_stock"],
    name="Healthy Stock",
    marker_color="#2ECC71"
))
fig.add_trace(go.Bar(
    x=df["warehouse_country"],
    y=df["low_stock"],
    name="Low Stock ⚠️",
    marker_color="#E74C3C"
))
fig.update_layout(
    barmode="stack",
    title="⚠️ Low Stock vs Healthy Stock by Country",
    xaxis_title="Country",
    yaxis_title="Number of Products",
    legend_title="Stock Health"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        product_name,
        ROUND(SUM(inventory_value), 2) AS total_inventory_value,
        SUM(stock_on_hand)             AS total_stock_on_hand,
        inventory_status
    FROM inventory_health
    GROUP BY product_name, inventory_status
    ORDER BY total_inventory_value DESC
    LIMIT 15
""").toPandas()

fig = px.bar(
    df,
    x="total_inventory_value",
    y="product_name",
    orientation="h",
    title="💰 Top 15 Products by Inventory Value",
    color="inventory_status",
    color_discrete_map={
        "in_stock":     "#2ECC71",
        "low_stock":    "#F39C12",
        "out_of_stock": "#E74C3C",
        "overstocked":  "#3498DB"
    },
    text="total_inventory_value"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Inventory Value ($)",
    yaxis_title="Product"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        warehouse_country,
        SUM(stock_on_hand)    AS total_on_hand,
        SUM(stock_reserved)   AS total_reserved,
        SUM(available_stock)  AS total_available
    FROM inventory_health
    GROUP BY warehouse_country
    ORDER BY warehouse_country
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(name="Stock On Hand",  x=df["warehouse_country"], y=df["total_on_hand"],   marker_color="#3498DB"))
fig.add_trace(go.Bar(name="Stock Reserved", x=df["warehouse_country"], y=df["total_reserved"],  marker_color="#E74C3C"))
fig.add_trace(go.Bar(name="Available",      x=df["warehouse_country"], y=df["total_available"], marker_color="#2ECC71"))

fig.update_layout(
    barmode="group",
    title="📊 Stock On Hand vs Reserved vs Available by Country",
    xaxis_title="Country",
    yaxis_title="Units",
    legend_title="Stock Type"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        product_name,
        warehouse_country,
        available_stock,
        reorder_point,
        inventory_status,
        (reorder_point - available_stock) AS reorder_gap
    FROM inventory_health
    WHERE available_stock <= reorder_point
    ORDER BY reorder_gap DESC
    LIMIT 20
""").toPandas()

fig = px.bar(
    df,
    x="reorder_gap",
    y="product_name",
    orientation="h",
    title="🚨 Top 20 Products Needing Reorder (Gap = Reorder Point − Available)",
    color="warehouse_country",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    text="available_stock",
    hover_data=["reorder_point", "inventory_status"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Units Below Reorder Point",
    yaxis_title="Product"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        warehouse_country,
        warehouse_city,
        ROUND(SUM(inventory_value), 2) AS total_inventory_value,
        COUNT(DISTINCT product_id)     AS unique_products
    FROM inventory_health
    GROUP BY warehouse_country, warehouse_city
""").toPandas()

fig = px.treemap(
    df,
    path=["warehouse_country", "warehouse_city"],
    values="total_inventory_value",
    color="total_inventory_value",
    color_continuous_scale="Blues",
    title="🗺️ Inventory Value by Country → City (Treemap)"
)
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.show()